# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 (프롬프트 자유 수정)
- Cell 3+: 실험 (복사해서 쓰면 됨)

## 핵심 API
```python
a = Agent("이름", "system prompt")
a.say("메시지")                  # 단일 에이전트 응답
a.set_prompt("새 프롬프트")       # 프롬프트 변경
send(a, b, "메시지")             # A -> B 통신
chain([a, b, c], "메시지")       # A -> B -> C 체인
extract_number("답은 42입니다")   # 숫자 추출 -> 42.0
grade(got, expected)              # 채점 (10% 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
# Colab이면: !pip install -q torch transformers accelerate

from agents import load_model, Agent, send, chain, extract_number, grade

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: 에이전트 생성 (이름 + 프롬프트만 수정)
# ============================================================

a = Agent("Analyst", "You are a data analyst. Summarize the given data.")
b = Agent("Calculator", "Compute the requested value. Output only the number.")
# c = Agent("Verifier", "Check if correct. Reply CORRECT or INCORRECT.")

print(a, b)

---
## 기본 추론 테스트

In [ ]:
# 단일 에이전트
r = a.say("What is 15% of 200?")
print(r["response"], f"({r['tokens']}tok, {r['time']}s)")

In [ ]:
# A -> B 통신
send(a, b, "Data: employees=120, revenue=5M, profit=800K. Summarize.")

---
## 실험 템플릿

아래 셀들을 복사 후 프롬프트/데이터만 바꿔서 실험.

In [ ]:
# ---- 프롬프트 변경으로 조건 비교 ----
DATA = "revenue=10M, cost=6M, employees=500, debt=2M, equity=8M, cash=1M, tax=25%"

# Condition 1: No Context
a.set_prompt("Summarize all data comprehensively.")
r1 = a.say(f"Data: {DATA}")
print(f"[No Context] {r1['tokens']}tok")
print(f"  {r1['response'][:200]}\n")

# Condition 2: Mutual
a.set_prompt("Send only requested data as key=value. Nothing else.")
r2 = a.say(f"Data: {DATA}\nRecipient needs: debt, equity")
print(f"[Mutual] {r2['tokens']}tok")
print(f"  {r2['response'][:200]}")

In [ ]:
# ---- A -> B 파이프라인 + 채점 ----
DATA = "revenue=10M, cost=6M, employees=500, debt=2M, equity=8M"
TASK = "debt-to-equity ratio (debt / equity)"
EXPECTED = 0.25

# A가 요약
a.set_prompt("Summarize the data.")
a_out = a.say(f"Data: {DATA}")

# B가 계산
b.set_prompt("Compute the value. Output only a number.")
b_out = b.say(f"Info: {a_out['response']}\nTask: {TASK}")

got = extract_number(b_out["response"])
ok = grade(got, EXPECTED)
print(f"A: {a_out['tokens']}tok | B: {got} (expected {EXPECTED}) {'OK' if ok else 'FAIL'}")

In [ ]:
# ---- 3-Agent Chain ----
a = Agent("Analyst", "Summarize key financial metrics.")
b = Agent("Calculator", "Compute profit margin = (revenue-cost)/revenue. Show work.")
c = Agent("Verifier", "Check if correct. Reply CORRECT or INCORRECT.")

result = chain([a, b, c], "Data: revenue=10M, cost=6M, debt=2M, equity=8M")
print(f"\nFinal: {result['final'][:200]}")
print(f"Total tokens: {result['total_tokens']}")